## Exploration du dataset allociné

***Analyse exploratoire des critique de films françaises.***

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns # Pour des visualisations plus claires
import torch # Indispensable pour la gestion du matériel et des graines 
from datasets import load_dataset
from transformers import AutoTokenizer # Nécessaire pour l'analyse de fragmentation P03 

# --- RIGUEUR MÉTHODOLOGIQUE ET REPRODUCTIBILITÉ
# Fixer les graines est crucial pour la validité scientifique de votre comparaison P03
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- OPTIMISATION POUR CONFIGURATION CPU
# Recommandation technique pour stabiliser les calculs sur processeur
if not torch.cuda.is_available():
    torch.set_num_threads(4) 
    print(f"Optimisation CPU : {torch.get_num_threads()} threads configurés pour l'exécution.")

## 1.Chargement

In [ ]:
dataset = load_dataset('allocine')
df = pd.DataFrame(dataset['train'])
df['label_name'] = df['label'].map({0: 'négatif', 1: 'positif'})
print(f"Total : {len(df)} critiques")
df.head()

## 2. Distribution des classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
axes[0].bar(['Négatif', 'Positif'], [counts[0], counts[1]], color=['#ef4444','#22c55e'])
axes[0].set_title('Distribution des classes')
for i,v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v+200, str(v), ha='center', fontweight='bold')

df['length'] = df['review'].apply(len)
for lbl, color, name in [(0,'#ef4444','Négatif'),(1,'#22c55e','Positif')]:
    axes[1].hist(df[df['label']==lbl]['length'], bins=50, alpha=0.6, color=color, label=name)
axes[1].set_title('Longueur des critiques (caractères)')
axes[1].legend()
plt.tight_layout()
plt.savefig('../figures/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Exemple de critique

In [ ]:
print('=== CRITIQUES POSITIVES ===')
for txt in df[df['label']==1]['review'].head(3):
    print(f'  {txt[:200]}\n')

print('=== CRITIQUES NÉGATIVES ===')
for txt in df[df['label']==0]['review'].head(3):
    print(f'  {txt[:200]}\n')

## 4. Analyse des tokenisers

In [ ]:
import sys
sys.path.append('..')
from src.data_loader import analyze_tokenizer_comparison
from transformers import AutoTokenizer

tok_db = AutoTokenizer.from_pretrained('distilbert-base-uncased')
tok_cb = AutoTokenizer.from_pretrained('camembert-base')

df_tok = analyze_tokenizer_comparison(tok_db, tok_cb)
display(df_tok[['phrase','n_mots','n_tokens_distilbert','n_tokens_camembert','ratio_distilbert','ratio_camembert']])